# Train "Hey Jarvis" wake-word classifier (1-click)

Runs entirely on a Colab GPU (T4/L4). Produces `hey_jarvis.onnx` (input `(1,16,96)` → score `(1,1)`)
which you drop into:
```
jarvis/android/app/src/main/assets/wakeword/hey_jarvis.onnx
```
then rebuild the APK. The two frozen front-end models are reused as-is.

Tips for better accuracy (optional — edit the YAML in the next cell):
- `n_samples: 500` → bump to `3000`+ for more speaker diversity
- `steps: 8000` → bump to `20000`+ for a tighter model
- Add your own recordings under `data/` for real-device hard negatives

In [ ]:
%%writefile hey_jarvis.yaml
# "Hey Jarvis" wake-word training config (livekit-wakeword 0.2.1)
model_name: hey_jarvis
target_phrases:
  - "hey jarvis"
  - "hey javis"
  - "hey jarvis assistant"

n_samples: 500
n_samples_val: 100
n_background_samples: 200
n_background_samples_val: 40
tts_batch_size: 32
tts_backend: piper_vits

noise_scales: [0.98]
noise_scale_ws: [0.98]
length_scales: [0.75, 1.0, 1.25]
splerp_weights: [0.2, 0.35, 0.5, 0.65, 0.8]

custom_negative_phrases:
  - "hey jerry"
  - "hey charlie"
  - "hey garrison"
  - "jarvis"
  - "hey siri"
  - "ok google"
  - "hey alexa"
  - "hey sam"
  - "hey carlos"

data_dir: ./data
output_dir: ./output

augmentation:
  clip_duration: 2.0
  batch_size: 16
  rounds: 1
  background_paths: ["./data/backgrounds"]
  rir_paths: ["./data/rirs"]

model:
  model_type: conv_attention
  model_size: small

steps: 8000
learning_rate: 1.0e-4
weight_decay: 1.0e-2
label_smoothing: 0.05
max_negative_weight: 1500.0
target_fp_per_hour: 0.2

batch_n_per_class:
  positive: 24
  adversarial_negative: 24
  ACAV100M_sample: 0
  background_noise: 24


In [ ]:
# 1) Install livekit-wakeword (train/eval/export extras)
!pip install -q "livekit-wakeword[train,eval,export]"

# 2) Fetch frozen models + Piper weights + backgrounds/RIRs
!livekit-wakeword setup --config hey_jarvis.yaml

# 3) Synthesize positive + adversarial-negative speech
!livekit-wakeword generate hey_jarvis.yaml

# 4) Augment (noise/RIR mixing) + extract frozen-model features
!livekit-wakeword augment hey_jarvis.yaml

# 5) Train (long step — Colab L4 ~30-60 min). Tail the log to watch EER.
!livekit-wakeword train hey_jarvis.yaml

# 6) Export ONNX classifier
!livekit-wakeword export hey_jarvis.yaml

import os
out = "output/hey_jarvis/hey_jarvis.onnx"
print("EXPORTED:", out, os.path.getsize(out), "bytes")

In [ ]:
# Download the trained classifier to your local machine, then copy it to:
#   jarvis/android/app/src/main/assets/wakeword/hey_jarvis.onnx
# and rebuild the APK.
from google.colab import files
files.download("output/hey_jarvis/hey_jarvis.onnx")